# Introducción a la limpieza de datos

Los datos obtenidos de sistemas reales suelen requerir trabajo antes de poder analizarlos. Es común encontrar celdas vacías, formatos incompatibles, fechas guardadas como texto, categorías escritas de distintas maneras, registros repetidos o valores que no encajan con el contexto.

Preparar datos no consiste únicamente en borrar filas. Implica inspeccionar la información, interpretar sus defectos, elegir un tratamiento justificable, aplicarlo y comprobar sus efectos.

En este recorrido aprenderemos a reconocer problemas frecuentes y a seguir un proceso reproducible para diagnosticar, transformar y revisar un dataset.

Al terminar podrás:

- Describir el propósito de la limpieza de datos.
- Identificar problemas habituales en fuentes reales.
- Separar el diagnóstico de la decisión de tratamiento.
- Evaluar cuándo conviene conservar, corregir o eliminar información.
- Comprobar que una transformación produjo el resultado esperado.
- Aplicar el ciclo: inspeccionar, decidir, transformar y validar.

La calidad de un análisis depende en buena medida de la calidad de la entrada. Un modelo, una métrica o una gráfica pueden parecer correctos y aun así estar afectados por datos incompletos o mal interpretados. Por eso la limpieza debe considerarse una parte del razonamiento analítico, no una tarea secundaria que se realiza al final.

Durante el notebook se trabajará con evidencia observable. Primero se conocerá la fuente, luego se describirán sus señales de calidad y finalmente se preparará el terreno para aplicar correcciones justificadas.

## El dataset de práctica

Trabajaremos con **Cafe Sales — Dirty Data for Cleaning Training**, un conjunto de datos de ventas que el usuario cargará directamente en Colab.

Cada registro representa una transacción de cafetería. Sus variables incluyen el artículo, la cantidad, el precio unitario, el importe total, el medio de pago, la ubicación y la fecha.

La fuente es útil porque contiene señales de calidad imperfecta: ausencias, valores especiales, formatos que requieren conversión y posibles inconsistencias. La meta no es memorizar comandos de Pandas, sino aprender a preparar una fuente imperfecta para analizarla con mayor confianza.

Antes de modificar cualquier columna conviene responder:

```text
¿Qué anomalías están presentes?
¿Qué variables requieren atención?
¿Qué representa cada valor vacío o especial?
¿Pandas reconoció correctamente los tipos?
¿Existe una relación entre columnas que permita comprobar resultados?
¿Qué tratamiento es razonable y cómo se comprobará?
```

Métodos como `dropna()`, `fillna()` o `replace()` son herramientas, no decisiones. La transformación debe responder a un problema observado y acompañarse de una revisión posterior.

El hecho de que el dataset sea realista es importante: en una tabla preparada artificialmente normalmente conocemos de antemano qué valores están bien y cuáles están mal. En cambio, aquí tendremos que inferir el significado de las señales a partir de las columnas, sus relaciones y la distribución de los registros.

La descarga también será útil para practicar un flujo reproducible en Google Colab. Cualquier persona que ejecute las celdas podrá obtener la fuente, inspeccionarla y repetir las mismas comprobaciones, siempre que tenga acceso a internet y a las dependencias requeridas.

## La limpieza empieza con una pregunta

Al conocer Pandas es fácil asociar la limpieza con métodos como:

```text
dropna()
fillna()
replace()
astype()
to_datetime()
drop_duplicates()
```

Todos pueden ser útiles, pero el primer paso no es escribir código. Es aclarar qué se intenta corregir.

Un valor ausente en el medio de pago no tiene necesariamente la misma importancia que uno ausente en la cantidad o en la fecha. Para elegir una solución necesitamos conocer el significado del campo, la causa probable de la ausencia y el impacto que tendría cada alternativa.

El trabajo puede resumirse así:

```text
examinar el problema
        ↓
seleccionar un tratamiento
        ↓
realizar el cambio
        ↓
comprobar el resultado
```

Sin diagnóstico podemos aplicar una corrección equivocada; sin verificación podemos confundir un cambio con una mejora. Por eso cada transformación debe tener una razón explícita y una comprobación asociada.

El mismo comando puede ser apropiado en una situación y perjudicial en otra. Eliminar filas puede reducir el ruido, pero también puede quitar observaciones valiosas; completar datos puede facilitar los cálculos, pero puede crear valores que nunca fueron observados. La herramienta es la misma, mientras que el criterio cambia según el caso.

Por ello, una transformación bien explicada debe dejar claro qué señal la motivó, qué supuesto utiliza y qué revisión permitirá saber si fue adecuada.

## Clases de problemas que aparecen en datos reales

Algunas anomalías se ven en una muestra; otras solo se hacen evidentes al calcular, filtrar o graficar.

- **Ausencias:** `NaN` puede alterar conteos, promedios y visualizaciones, aunque no siempre deba eliminarse.
- **Registros repetidos:** una fila duplicada puede ser un error o una observación válida que comparte características con otra.
- **Categorías no normalizadas:** `"Credito"`, `"credito"`, `"Crédito"` y `" credito "` pueden significar lo mismo para una persona, pero Pandas los separa.
- **Formatos incorrectos:** importes y cantidades pueden llegar como texto; una fecha puede quedar como `object`.
- **Valores dudosos:** cantidades negativas, precios cero, fechas fuera de rango o totales que no coinciden con sus componentes necesitan investigación.
- **Problemas de estructura:** nombres ambiguos, columnas redundantes o varios datos mezclados en un mismo campo dificultan el análisis.

| Situación | Ejemplo | Consecuencia posible |
|---|---|---|
| Ausencia | No hay medio de pago | Complica el análisis por categoría |
| Repetición | Una venta aparece dos veces | Incrementa artificialmente los totales |
| Categoría variable | `"Credito"` y `"credito"` | Divide los agrupamientos |
| Tipo inadecuado | Precio almacenado como texto | Impide operar con números |
| Fecha como texto | Fecha no convertida | Dificulta ordenar por tiempo |
| Valor sospechoso | Precio igual a cero | Puede señalar una carga defectuosa |
| Nombre ambiguo | `col1` o `dato` | Reduce la comprensión del dataset |

Reconocer una señal no determina por sí mismo la solución; la decisión requiere contexto.

La presencia de una de estas situaciones no demuestra automáticamente que exista un error. Un precio cero podría representar una promoción, una fecha futura podría corresponder a una reserva y dos registros iguales podrían ser transacciones independientes si no existe un identificador único. La limpieza requiere distinguir entre anomalía, excepción válida y error confirmado.

En las siguientes etapas, estas categorías servirán como mapa de exploración. La tabla no prescribe una solución; ayuda a recordar qué evidencia debe buscarse antes de intervenir.

## Primero observar, después modificar

Ante un valor vacío o una categoría extraña, la reacción más rápida suele ser corregirla. Sin embargo, antes conviene medir el alcance, explorar su distribución y considerar su causa.

Por ejemplo, no tiene el mismo significado que falte el 1% de una variable auxiliar que el 60% de una variable indispensable. Tampoco es equivalente una ausencia dispersa a una concentrada en una fecha, sucursal o producto.

La misma cautela se aplica a los duplicados: dos ventas pueden compartir producto, precio y fecha sin ser el mismo evento. El diagnóstico evita automatizar una decisión que podría eliminar información válida.

La pregunta útil no es:

```text
¿Cómo elimino esta anomalía?
```

sino:

```text
¿Qué significa esta anomalía dentro del dataset?
```

Con esa respuesta se puede valorar mejor el tratamiento. En este trabajo, comprender el problema es más importante que corregirlo rápidamente.

Un diagnóstico útil combina conteos, ejemplos y contexto. El conteo indica el tamaño del fenómeno; los ejemplos muestran cómo está representado; y el contexto ayuda a valorar si afecta una conclusión importante. Usar solo una de esas perspectivas puede llevar a una interpretación incompleta.

Esta forma de trabajo es especialmente importante cuando el dataset se utilizará para aprendizaje automático. Una corrección aparentemente pequeña puede modificar la distribución de las variables o introducir sesgos en las observaciones que después recibe el modelo.

## Elegir el tratamiento adecuado

Una vez medido el problema, hay que decidir cómo actuar. Para valores faltantes, por ejemplo, se puede conservar el registro, eliminar filas o columnas, completar con una regla estadística o reconstruir el valor usando otras variables.

La alternativa correcta depende del objetivo, la cantidad de información afectada, la importancia de la columna y el contexto de negocio. Un medio de pago desconocido quizá pueda conservarse como tal; una cantidad o un precio ausente puede impedir calcular el importe.

Las relaciones internas del dataset también pueden ayudar. Si están disponibles `Quantity`, `Price Per Unit` y `Total Spent`, un dato ausente podría reconstruirse con los otros dos, siempre que la relación sea válida.

En cuanto a duplicados, dos filas idénticas pueden ser una copia accidental, pero dos ventas parecidas no deben eliminarse sin una regla que demuestre que representan el mismo evento.

Antes de actuar, conviene documentar:

```text
qué se detectó
qué columnas y filas afecta
qué impacto puede tener
qué tratamientos son posibles
qué información conserva cada opción
qué riesgo introduce la decisión
cómo se comprobará el resultado
```

Así, la limpieza queda sustentada por decisiones trazables y no por una secuencia automática de comandos.

La estrategia también debe considerar qué análisis se realizará después. Una columna necesaria para calcular ventas puede requerir un control más estricto que una variable usada únicamente para describir segmentos. Del mismo modo, una imputación aceptable para una visualización exploratoria podría no ser suficiente para un modelo predictivo.

Documentar la alternativa descartada también puede ser valioso. Así, otra persona podrá entender por qué se conservaron ciertos registros, por qué se marcaron como desconocidos o por qué se decidió esperar hasta contar con más evidencia.

## Comprobar cada transformación

Una operación no está terminada cuando se ejecuta sin error. Está terminada cuando su efecto coincide con lo esperado y no introduce una nueva inconsistencia.

Después de unificar categorías, conviene revisar los valores y sus frecuencias. Tras convertir una columna a número, hay que comprobar qué valores quedaron sin convertir. Si se quitaron duplicados, se comparan los tamaños antes y después; si se imputaron ausencias, se revisa su nuevo conteo y la coherencia de los valores generados.

La pregunta de control es:

```text
¿Qué evidencia demuestra que la limpieza funcionó?
```

| Recurso | Uso de comprobación |
|---|---|
| `head()` | Revisar filas después del cambio |
| `info()` | Confirmar tipos y valores no nulos |
| `isna().sum()` | Medir ausencias restantes |
| `value_counts()` | Revisar frecuencias de categorías |
| `unique()` | Inspeccionar valores distintos |
| `duplicated().sum()` | Contar repeticiones |
| `describe()` | Revisar estadísticas numéricas |
| `shape` | Comparar dimensiones |

Validar convierte la limpieza en un proceso de control de calidad: el dataset debe quedar más confiable, no simplemente distinto.

La verificación no tiene que ser compleja para ser útil. Un conteo antes y después, una comparación de tipos o una pequeña muestra de valores transformados pueden revelar rápidamente si la operación produjo el efecto esperado. Lo importante es que la evidencia sea específica para el problema tratado.

También conviene revisar los casos límite. Una conversión puede funcionar para la mayoría de las filas y fallar precisamente en los valores especiales que originaron el problema. Esos casos deben quedar visibles para que la siguiente decisión no se base en una falsa sensación de limpieza.

## Secuencia de trabajo

Cada fuente exige decisiones propias, pero una ruta ordenada ayuda a evitar cambios improvisados.

```text
importar la fuente
        ↓
conocer su estructura
        ↓
localizar señales de calidad
        ↓
definir prioridades
        ↓
transformar lo necesario
        ↓
revisar los efectos
        ↓
dejar constancia de las decisiones
        ↓
preparar la versión analizable
```

El proceso puede repetirse. Una conversión numérica puede revelar textos ocultos; una revisión de categorías puede descubrir valores no previstos; el análisis de duplicados puede exigir una combinación de columnas más completa.

Por eso no es una receta rígida, sino un ciclo:

```text
observar → decidir → transformar → verificar
```

Aplicaremos este ciclo gradualmente sobre el dataset de ventas, atendiendo cada señal por separado.

La documentación forma parte del resultado porque una tabla limpia no siempre conserva las razones de los cambios realizados. Registrar qué se observó y qué se decidió facilita reproducir el proceso, comparar versiones y explicar los resultados a otras personas.

La secuencia también permite detenerse en cualquier punto. Si una inspección revela que la fuente no es suficiente para tomar una decisión, es preferible documentar la incertidumbre que aplicar una corrección irreversible sin respaldo.

## Cargar un CSV desde Google Colab

En esta versión, el usuario seleccionará manualmente el archivo CSV desde su equipo mediante el selector de archivos de Google Colab.

Al ejecutar la celda de carga aparecerá un botón para elegir el archivo. Colab lo almacenará temporalmente en la sesión y después podremos leerlo con Pandas. Sube un archivo con extensión `.csv` antes de continuar.

Este procedimiento permite trabajar con el archivo que el usuario elija, sin depender de una plataforma externa.

In [1]:
# Carga manual del archivo en Google Colab

from google.colab import files
import io
import pandas as pd

# El usuario selecciona desde su equipo el CSV que desea analizar.
uploaded = files.upload()
archivos = list(uploaded.keys())
archivos_csv = [archivo for archivo in archivos if archivo.lower().endswith(".csv")]

if not archivos_csv:
    raise ValueError("Debes subir al menos un archivo con extensión .csv")

nombre_csv = archivos_csv[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre_csv]))

df.head()

Saving dirty_cafe_sales.csv to dirty_cafe_sales.csv


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# Registramos los nombres de los archivos recibidos.

archivos = list(uploaded.keys())

archivos

['dirty_cafe_sales.csv']

El archivo seleccionado ya está cargado en el `DataFrame` `df`.

La salida de `head()` permite confirmar que Pandas pudo leer el CSV. Después continuaremos con la revisión de dimensiones, columnas, tipos, ausencias y posibles inconsistencias.

## Exploración inicial

Con el archivo cargado, comenzamos por describir su forma antes de aplicar cualquier corrección.

Revisaremos dimensiones, nombres de columnas, tipos inferidos y primeras señales de calidad. La primera comprobación será el tamaño del `DataFrame`.

La inspección inicial funciona como una línea base. Si después se aplican transformaciones, podremos comparar el estado posterior con estas dimensiones, tipos y conteos iniciales. Sin esa referencia, es más difícil determinar qué cambió realmente.

También ayuda a separar dos momentos del trabajo: conocer la fuente y corregirla. En este capítulo nos enfocaremos primero en el conocimiento de la fuente.

In [4]:
df.shape

(10000, 8)

`shape` devuelve el número de filas y columnas.

Como se trata de una fuente real, la inspección debe apoyarse en resúmenes: revisar manualmente cada registro no es una estrategia viable para un dataset de este tamaño.

Las dimensiones ofrecen una referencia para interpretar todos los conteos posteriores. Por ejemplo, el número de ausencias adquiere significado cuando se compara con el total de filas, y la cantidad de columnas ayuda a evaluar si la estructura es manejable o si requiere una revisión más detallada.

Esta comprobación es breve, pero establece el tamaño sobre el que se evaluarán las decisiones de limpieza.

In [5]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

`columns` enumera las variables disponibles.

Sus nombres permiten interpretar el contenido general: identificador de transacción, artículo, cantidad, precio unitario, importe, método de pago, ubicación y fecha.

Los nombres también sirven para detectar posibles problemas de diseño. Una variable con un nombre ambiguo puede requerir consultar la documentación de la fuente, mientras que una variable claramente nombrada facilita construir reglas de validación y comunicar los resultados.

Por ahora utilizaremos esta lista para orientar las siguientes inspecciones.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


`info()` resume la estructura: muestra los valores no nulos y el tipo que Pandas asignó a cada columna.

Estos datos ayudan a ubicar ausencias y a detectar campos numéricos almacenados como texto. Todavía no corregimos nada; primero reunimos evidencia para decidir qué revisar.

La diferencia entre el tipo esperado y el tipo observado no debe interpretarse de forma aislada. Es necesario mirar valores concretos para saber si el problema proviene de símbolos, cadenas especiales, separadores decimales o valores vacíos.

Esta lectura de `info()` prepara la comparación con las muestras y los conteos que se ejecutarán enseguida.

## Buscar indicios de calidad

Después de conocer la estructura, iniciamos una búsqueda dirigida de anomalías. Aún no transformaremos el dataset; registraremos señales para analizarlas con más detalle.

Algunas preguntas de control son:

```text
¿Existen celdas vacías?
¿Hay números representados como texto?
¿Las fechas tienen un formato aprovechable?
¿Las categorías contienen valores inesperados?
¿Una variable puede servir para validar otra?
```

Comenzaremos observando una muestra de filas.

Estas preguntas convierten la exploración en una revisión ordenada. En lugar de mirar la tabla sin un objetivo, cada consulta se relaciona con una posible fuente de error y con una decisión que se podrá tomar más adelante.

La muestra siguiente no resolverá las preguntas por sí sola, pero permitirá identificar patrones visibles y orientar las comprobaciones globales.

In [7]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


La muestra ayuda a entender la forma de los registros, pero no representa necesariamente todo el archivo.

Una anomalía puede aparecer lejos de las primeras filas o afectar solo a una fracción pequeña. Por ello combinaremos ejemplos con resúmenes globales.

La combinación de inspección local y resumen global es más robusta que cualquiera de las dos técnicas por separado. Las filas concretas ayudan a interpretar los valores, mientras que las funciones de resumen muestran si el patrón es aislado o generalizado.

Esta distinción será relevante cuando aparezcan valores especiales que solo se presentan en una parte de la tabla.

In [8]:
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


`describe(include="all")` reúne estadísticas de variables numéricas y categóricas.

No es necesario interpretar cada celda en esta etapa. Buscamos pistas: frecuencias inusuales, gran variedad de valores, posibles ausencias, tipos inesperados o columnas que requieran una inspección especializada.

El diagnóstico comienza leyendo esas señales antes de aplicar una transformación.

El resumen debe leerse como un mapa de preguntas, no como una conclusión definitiva. Una frecuencia alta no implica que un valor sea correcto, y una frecuencia baja no implica que sea un error. Cada señal requiere volver a las filas y al significado de la variable.

Con esta perspectiva, `describe()` se convierte en una herramienta para decidir qué investigar, no en un sustituto del análisis.

## Medir valores ausentes

La presencia de información faltante es una de las primeras condiciones que conviene cuantificar.

Pandas suele representarla mediante `NaN`. En esta fase no decidiremos si conservar, completar o eliminar esos valores; únicamente mediremos en qué columnas aparecen y cuál es su magnitud.

Para iniciar el conteo utilizaremos:

Los nulos pueden producir resultados distintos según la operación realizada. Algunas funciones los ignoran, otras los cuentan o los propagan, y los filtros pueden excluir filas sin que el usuario lo note. Medirlos explícitamente evita que ese comportamiento quede oculto.

El conteo que sigue será la referencia inicial para comparar cualquier tratamiento futuro.

In [9]:
df.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


`isna()` marca las celdas vacías y `sum()` las agrega por columna.

El resultado muestra dónde se concentran las ausencias. Es una señal de diagnóstico, no una instrucción automática de eliminación.

También calcularemos su proporción para comparar columnas con tamaños de problema distintos:

El conteo por columna permite priorizar. Una variable con pocas ausencias puede requerir una revisión diferente de otra donde la información faltante sea dominante. Aun así, la cantidad no es el único criterio: también importa qué papel cumple la columna.

El porcentaje complementará la medición porque permite comparar el problema entre columnas con diferentes niveles de completitud.

In [10]:
porcentaje_faltantes = df.isna().mean() * 100

porcentaje_faltantes

,0
Transaction ID,0.00
Item,3.33
Quantity,1.38
Price Per Unit,1.79
Total Spent,1.73
Payment Method,25.79
Location,32.65
Transaction Date,1.59


El porcentaje pone el conteo en contexto.

Cinco valores ausentes tienen una importancia distinta en una tabla de 20 filas que en una de 10.000. Por eso conviene conservar tanto el número absoluto como la proporción.

Más adelante se evaluará el tratamiento; por ahora solo incorporamos esta medición al diagnóstico.

La proporción no reemplaza al conteo absoluto. Ambos valores deben leerse juntos: el porcentaje ayuda a dimensionar el problema, mientras que el conteo permite conocer cuántos registros concretos quedarían involucrados en una decisión.

Esta distinción será útil cuando se evalúen estrategias de eliminación, conservación o imputación.

## Ausencias representadas como texto

`isna()` reconoce valores nulos reales, como `NaN`, pero una fuente también puede usar textos para indicar que no hay información:

```text
UNKNOWN
ERROR
N/A
Sin dato
No informado
```

Si esos marcadores se cargan como cadenas, Pandas los cuenta como categorías normales. Por eso el diagnóstico debe incluir una revisión de valores únicos en las variables categóricas.

Los marcadores textuales pueden variar entre fuentes y, en ocasiones, mezclarse con nulos reales dentro de la misma columna. Si se ignoran, los conteos de categorías pueden incluir una falsa categoría y los cálculos de frecuencia pueden quedar distorsionados.

Identificar estas representaciones es el primer paso; convertirlas en nulos o conservarlas como categorías dependerá del significado que tengan en el contexto de la fuente.

In [11]:
df["Item"].value_counts(dropna=False)

,count
Item,
Juice,1171
Coffee,1165
Salad,1148
Cake,1139
Sandwich,1131
Smoothie,1096
Cookie,1092
Tea,1089
UNKNOWN,344


In [12]:
df["Payment Method"].value_counts(dropna=False)

,count
Payment Method,
NaN,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


In [13]:
df["Location"].value_counts(dropna=False)

,count
Location,
NaN,3265
Takeaway,3022
In-store,3017
ERROR,358
UNKNOWN,338


`value_counts(dropna=False)` incluye tanto las categorías como los nulos.

La salida puede revelar marcadores como `"UNKNOWN"` o `"ERROR"`, que quizá no aparezcan en `isna()` pero sí señalen información desconocida o una carga defectuosa.

La ausencia, por tanto, debe analizarse según su representación en la fuente. En esta etapa registramos esos valores; el tratamiento se decidirá después.

El argumento `dropna=False` es intencional: necesitamos ver la diferencia entre un nulo real y una cadena que cumple una función parecida. Esta revisión permite no confundir la forma de almacenamiento con el significado del dato.

Más adelante, cualquier normalización deberá comprobar que no se mezclaron categorías válidas con marcadores de ausencia.

## Revisar los tipos de columna

Además de las ausencias, debemos comprobar si Pandas interpretó cada variable con el tipo adecuado.

En una tabla de ventas, `Quantity`, `Price Per Unit` y `Total Spent` deberían admitir operaciones numéricas. `Transaction Date` debería poder convertirse en una fecha.

La revisión se realizará con:

El tipo de dato influye en las operaciones disponibles y en la forma en que Pandas procesa la columna. Una cantidad como texto no se comporta igual que una cantidad numérica, y una fecha como cadena no ofrece las mismas posibilidades de ordenamiento, extracción o agrupamiento temporal.

Revisar los tipos antes de convertirlos reduce el riesgo de aplicar una función a una variable que no contiene el formato esperado.

In [14]:
df.dtypes

,0
Transaction ID,object
Item,object
Quantity,object
Price Per Unit,object
Total Spent,object
Payment Method,object
Location,object
Transaction Date,object


Una variable que debería ser numérica y aparece como `object` puede contener texto, símbolos, valores especiales o errores.

Las fechas también pueden llegar como cadenas. Eso no confirma que estén mal, pero sí indica que habrá que convertirlas antes de analizarlas temporalmente.

En este punto solo identificaremos candidatos a revisión. A continuación observaremos algunas columnas concretas.

La conversión debe realizarse con cuidado porque un error silencioso puede transformar valores problemáticos en nulos o descartar información. Antes de convertir, conviene conocer qué formatos aparecen y definir cómo se revisarán los casos que no puedan interpretarse.

La muestra específica permitirá conectar el tipo general de `dtypes` con los valores reales almacenados en las columnas.

In [15]:
df[["Quantity", "Price Per Unit", "Total Spent", "Transaction Date"]].head()

,Quantity,Price Per Unit,Total Spent,Transaction Date
0,2,2.0,4.0,2023-09-08
1,4,3.0,12.0,2023-05-16
2,4,1.0,ERROR,2023-07-19
3,2,5.0,10.0,2023-04-27
4,2,2.0,4.0,2023-06-11


La vista permite detectar valores que podrían explicar una inferencia de tipo incorrecta: cadenas, símbolos, errores o celdas vacías.

En una etapa posterior se podrían usar `pd.to_numeric()` y `pd.to_datetime()` con controles adecuados. Por ahora, basta con ubicar las variables que requieren ese tratamiento.

Una inspección de ejemplos puede revelar detalles que no aparecen en el resumen de tipos: espacios, símbolos monetarios, textos de error o formatos de fecha mezclados. Esos detalles determinan qué estrategia de conversión será segura.

La conversión quedará para una etapa posterior, cuando el problema esté suficientemente descrito y podamos validar sus resultados.

## Relaciones internas para validar datos

Algunas variables contienen una relación que sirve como comprobación. En este caso esperamos que:

```text
Total Spent = Quantity × Price Per Unit
```

Con dos componentes disponibles, el tercero podría reconstruirse; si los tres existen y no coinciden, habría una inconsistencia que investigar.

Todavía no corregiremos valores. Solo identificaremos esta regla interna porque más adelante puede ayudar a validar o reconstruir información.

Revisemos algunas filas con esas variables:

Las reglas internas son una forma de control de consistencia. No garantizan por sí solas que todos los valores sean correctos, pero permiten detectar desacuerdos entre variables que deberían describir el mismo evento económico.

Antes de reconstruir un dato habría que comprobar también unidades, redondeos y posibles excepciones comerciales. Por ahora solo dejamos identificada la relación que merece ser analizada.

In [16]:
df[["Quantity", "Price Per Unit", "Total Spent"]].head(10)

,Quantity,Price Per Unit,Total Spent
0,2,2.0,4.0
1,4,3.0,12.0
2,4,1.0,ERROR
3,2,5.0,10.0
4,2,2.0,4.0
5,5,4.0,20.0
6,3,3.0,9.0
7,4,4.0,16.0
8,5,3.0,15.0
9,5,4.0,20.0


Esta muestra introduce una posible validación basada en la propia fuente.

No siempre necesitamos una regla externa: las relaciones entre columnas pueden servir para detectar errores, reconstruir ausencias y evaluar transformaciones.

La idea será especialmente útil al estudiar conversiones numéricas y valores faltantes.

Una regla de este tipo puede utilizarse antes y después de una transformación. Primero ayuda a detectar qué registros son sospechosos; después permite comprobar si la versión preparada conserva la coherencia entre sus columnas.

La validación interna es especialmente valiosa cuando no existe una fuente externa sencilla contra la cual comparar cada registro.

## Inventario inicial de hallazgos

Después de explorar el archivo, conviene dejar una lista provisional de señales. No es todavía el diagnóstico final, sino una agenda para las siguientes etapas.

| Evidencia | Interpretación posible | Revisión siguiente |
|---|---|---|
| Nulos encontrados con `isna()` | Falta información en algunas celdas | Medir cantidad, proporción e importancia |
| `UNKNOWN` o `ERROR` | Marcadores de desconocido o error | Localizar columnas y definir tratamiento |
| Números con tipo `object` | Hay texto o valores no numéricos | Convertir controlando errores |
| Fechas como texto | No están listas para análisis temporal | Transformar al tipo fecha |
| Relación entre `Quantity`, `Price Per Unit` y `Total Spent` | Puede detectar diferencias o permitir reconstrucción | Comparar el total esperado con el observado |

Este inventario ayuda a ordenar el trabajo y evita transformar sin un motivo documentado. En las siguientes secciones se estudiarán primero las ausencias y después otros problemas como duplicados, categorías, tipos, fechas y validación.

Un inventario de hallazgos también facilita dividir el trabajo en etapas pequeñas. Cada elemento puede recibir una estrategia, una prioridad y una prueba de validación antes de pasar al siguiente.

La lista no afirma que todos los casos sean errores confirmados. Su función es conservar las preguntas abiertas y evitar que una señal observada durante la exploración se pierda.

## Síntesis

La limpieza de datos es una cadena de decisiones: observar la fuente, entender sus anomalías, elegir una estrategia, transformar lo necesario y comprobar el resultado.

Entre los problemas revisados se encuentran celdas ausentes, registros repetidos, categorías no normalizadas, tipos inadecuados, fechas como texto, valores sospechosos y estructuras poco claras.

Detectar una anomalía no determina automáticamente su tratamiento. Una ausencia puede eliminarse, conservarse, imputarse o reconstruirse según el contexto, el propósito del análisis y el impacto sobre los resultados.

La ruta de trabajo es:

```text
cargar → inspeccionar → detectar → priorizar → transformar → verificar → documentar
```

Aplicamos esta primera revisión al dataset **Cafe Sales — Dirty Data for Cleaning Training** usando `shape`, `columns`, `info()`, `head()`, `describe(include="all")`, `isna().sum()`, porcentajes de nulos, `value_counts(dropna=False)` y `dtypes`.

También identificamos la relación esperada entre `Quantity`, `Price Per Unit` y `Total Spent`, que podrá utilizarse como control interno.

```text
Una fuente limpia es una fuente diagnosticada, transformada y validada.
```

La primera inspección no pretendía producir una tabla definitiva, sino construir una base para las decisiones siguientes. Conocer el tamaño, los tipos, los valores especiales y las relaciones internas permite avanzar con mayor control.

En una limpieza profesional, esta evidencia inicial también sirve para comparar versiones del dataset y explicar qué se modificó, qué se conservó y qué problemas quedaron pendientes.

## Continuación

La siguiente sección profundizará en los valores ausentes: su representación en Pandas, las formas de contarlos, su proporción y las diferencias entre un nulo real, un marcador textual y un error de carga.

El objetivo será entender por qué una misma señal puede requerir tratamientos distintos según la variable y el contexto.

La ausencia puede tener significados diferentes: un dato que nunca se registró, un valor que no aplica, un marcador textual o un error de carga. Aprender a distinguirlos será necesario antes de decidir si se elimina, se recodifica, se completa o se conserva.

La siguiente etapa continuará el análisis usando el mismo principio: medir primero, elegir después y validar cualquier transformación.